In [ ]:
!pip install -q efficientnet
!pip install -q pandas scikit-learn
!pip install tensorflow

In [ ]:
!gdown https://drive.google.com/uc?id=1Os-7S1D9tfVSGZWXn4zOEuXMMnm9RH1Q

In [ ]:
!unzip -q dataset.zip

In [ ]:
import os
from PIL import Image, UnidentifiedImageError

def clean_image_folder(folder_path):
    """
    Walks through each subfolder in folder_path, attempts to open each file as an image,
    and deletes files that are not valid images.
    """
    for class_name in os.listdir(folder_path):
        class_dir = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):
            file_path = os.path.join(class_dir, filename)
            if os.path.isdir(file_path):
                continue

            try:
                # Try opening and verifying the image
                with Image.open(file_path) as img:
                    img.verify()
            except (UnidentifiedImageError, IOError, SyntaxError):
                print(f"Deleting invalid image: {file_path}")
                os.remove(file_path)

def main():
    base_dir = "dataset"
    for split in ("train", "val"):
        split_dir = os.path.join(base_dir, split)
        if not os.path.isdir(split_dir):
            print(f"Warning: {split_dir} does not exist, skipping.")
            continue
        clean_image_folder(split_dir)

main()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from pathlib import Path

# -------------------------------------------------
# 1.  Paths and basic hyper-parameters
# -------------------------------------------------

In [4]:
train_dir = Path("dataset/train")
val_dir   = Path("dataset/val")

IMAGE_SIZE  = (224, 224)     # EfficientNetB0 default
BATCH_SIZE  = 32
NUM_CLASSES = 6              # 5 breeds + NotCattle


In [4]:
train_dir = Path("dataset/train")
val_dir   = Path("dataset/val")

IMAGE_SIZE  = (224, 224)     # EfficientNetB0 default
BATCH_SIZE  = 32
NUM_CLASSES = 6              # 5 breeds + NotCattle


# -------------------------------------------------
# 2.  Data loaders with real-time augmentation
# -------------------------------------------------

In [ ]:
train_gen = ImageDataGenerator(
    rescale            = 1/255.,
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = "nearest"
)
val_gen = ImageDataGenerator(rescale = 1/255.)

train_ds = train_gen.flow_from_directory(
    train_dir, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)
val_ds   = val_gen.flow_from_directory(
    val_dir,   target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)

# Save class indices to a JSON file
import json
with open('class_indices.json', 'w') as f:
    json.dump(train_ds.class_indices, f)


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/train'

# -------------------------------------------------
# 3.  Model: EfficientNet-B0 backbone + custom head
# -------------------------------------------------

In [2]:
base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(*IMAGE_SIZE, 3))
x    = GlobalAveragePooling2D()(base.output)
out  = Dense(NUM_CLASSES, activation="softmax")(x)
model = Model(inputs=base.input, outputs=out)

NameError: name 'EfficientNetB0' is not defined

# -------------------------------------------------
# 4-A.  Warm-up – train only the new classifier
# -------------------------------------------------

In [ ]:
for layer in base.layers:
    layer.trainable = False

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ModelCheckpoint("best_efficientnet_cattle.h5", monitor="val_loss", save_best_only=True)
]

model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

NameError: name 'base' is not defined

# -------------------------------------------------
# 4-B.  Fine-tune – unfreeze all layers at low LR
# -------------------------------------------------

In [ ]:
for layer in base.layers:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

# -------------------------------------------------
# 5.  Save final checkpoint
# -------------------------------------------------

In [ ]:
model.save("final_efficientnet_cattle_breed.h5")
print("✅  Training complete. Best model: best_efficientnet_cattle.h5")